# Scorecard

Every number the official scoreboard reports, for each version, on the **seven open matches**. Each
match is damaged three ways and scored by a model that never saw it. Same scorer, same definitions
as `evaluation/run_evaluation.py`.

No data from the provided match is used, so these numbers are safe to share.

| Number | What it tells you |
|---|---|
| **spellF1** | Possessions found as whole possessions. **Ranked on this** |
| spellF1@0.3 … @0.7 | The same at each overlap level. A big drop from 0.3 to 0.7 means boundaries are off |
| **macroF1** | Right label on each frame, averaged over H, A and D. First tie-break |
| precision, recall, F1 per class | Where frame labels go wrong |
| alive_dead_f1 | Is the ball in play, ignoring team |
| team_acc | Right team, on frames both agree are live |
| trans_f1 at 0.5 s and 1 s | Each change of possession or stoppage, found close enough in time |
| mean_latency_frames | Early (negative) or late (positive) on average, at 25 frames a second |
| poss vs true_poss, fragmentation | Too many possessions (flicker) or too few (merged) |
| kappa, accuracy | Shown by the scorer, never ranked on |

---

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os, sys
sys.path[:0] = [os.path.abspath(p) for p in ('..', '../src', '../..', '../../starter', '../../evaluation')]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import score_external as bed
import possession_metrics as pm

## 1. The full card

Takes about a minute and a half.

In [ ]:
card, confusion = bed.scorecard()
card.round(3).T

## 2. Confusion matrices

Rows are the truth, columns are the prediction, shown as a share of each true row. A strong
version is dark on the diagonal. H↔A mix-ups are wrong team calls, and anything into or out of D
is a dead-ball mistake.

In [ ]:
names = list(card.index)
fig, axes = plt.subplots(1, len(names), figsize=(3.2 * len(names), 3.2))
for axis, name in zip(np.atleast_1d(axes), names):
    counts = confusion[name][:3, :3].astype(float)
    shares = counts / counts.sum(axis=1, keepdims=True)
    axis.imshow(shares, cmap='Blues', vmin=0, vmax=1)
    for r in range(3):
        for c in range(3):
            axis.text(c, r, '%.2f' % shares[r, c], ha='center', va='center',
                      color='white' if shares[r, c] > .5 else 'black', fontsize=9)
    axis.set_xticks(range(3)); axis.set_xticklabels('HAD')
    axis.set_yticks(range(3)); axis.set_yticklabels('HAD')
    axis.set_title(name, fontsize=9); axis.set_xlabel('predicted')
np.atleast_1d(axes)[0].set_ylabel('truth')
plt.tight_layout(); plt.show()

## 3. Are the possession edges in the right place?

spellF1 averages five overlap levels. A version whose score falls fast from 0.3 to 0.7 finds the
right possessions but puts their start or end in the wrong place, which is where touch-based timing
would help.

In [ ]:
ious = [c for c in card.columns if c.startswith('spellF1@')]
fig, ax = plt.subplots(figsize=(8, 4))
for name in names:
    ax.plot([float(c[-3:]) for c in ious], card.loc[name, ious], 'o-', label=name)
ax.set_xlabel('overlap needed to count a possession'); ax.set_ylabel('spellF1')
ax.grid(alpha=.3); ax.legend(fontsize=8); plt.show()

## 4. Per-class view and possession counts

In [ ]:
cols = ['precision_H', 'recall_H', 'f1_H', 'precision_A', 'recall_A', 'f1_A',
        'precision_D', 'recall_D', 'f1_D']
card[cols].round(3)

In [ ]:
card[['poss', 'true_poss', 'mean_poss_s', 'true_mean_poss_s', 'fragmentation',
      'trans_f1_0.5s', 'trans_f1_1s', 'mean_latency_frames']].round(2)